# Notebook 06: GSEA Preranked Analysis

Gene Set Enrichment Analysis using preranked gene lists against KEGG pathways.

**Input**: Ranked gene lists from each dataset
**Output**: Enriched pathways per dataset, convergent pathways

In [ ]:
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt

# Load ranked lists (from notebook 02)
nk_rnk = pd.read_csv('../tables/ranked_NK_IL2.rnk', sep='\t', header=None, index_col=0)
neut_tnf_rnk = pd.read_csv('../tables/ranked_Neut_TNF.rnk', sep='\t', header=None, index_col=0)
neut_lps_rnk = pd.read_csv('../tables/ranked_Neut_LPS.rnk', sep='\t', header=None, index_col=0)

print("Ranked lists loaded")

In [ ]:
# Run GSEA preranked for all datasets
datasets = {
    'NK_IL2': nk_rnk,
    'Neut_TNF': neut_tnf_rnk,
    'Neut_LPS': neut_lps_rnk
}

results = {}
for name, rnk in datasets.items():
    print(f"Running GSEA for {name}...")
    pre = gp.prerank(
        rnk=rnk,
        gene_sets='KEGG_2021_Human',
        threads=4,
        min_size=5,
        max_size=500,
        permutation_num=1000,
        outdir=None,
        seed=42,
        verbose=False
    )
    results[name] = pre
    sig = len(pre.res2d[pre.res2d['FDR q-val'] < 0.25])
    print(f"  Significant pathways (FDR<0.25): {sig}")
    pre.res2d.to_csv(f'../tables/GSEA_{name}_KEGG.csv', index=False)

In [ ]:
# Find convergent pathways (significant in >= 2 datasets)
nk_sig = results['NK_IL2'].res2d[results['NK_IL2'].res2d['FDR q-val'] < 0.25]['Term'].tolist()
tnf_sig = results['Neut_TNF'].res2d[results['Neut_TNF'].res2d['FDR q-val'] < 0.25]['Term'].tolist()
lps_sig = results['Neut_LPS'].res2d[results['Neut_LPS'].res2d['FDR q-val'] < 0.25]['Term'].tolist()

from functools import reduce
all_pathways = set(nk_sig) | set(tnf_sig) | set(lps_sig)

conv_data = []
for pw in all_pathways:
    in_nk = pw in nk_sig
    in_tnf = pw in tnf_sig
    in_lps = pw in lps_sig
    count = sum([in_nk, in_tnf, in_lps])
    if count >= 2:
        conv_data.append({'Pathway': pw, 'NK_IL2': in_nk, 'Neut_TNF': in_tnf, 
                         'Neut_LPS': in_lps, 'Datasets': count})

conv_df = pd.DataFrame(conv_data).sort_values('Datasets', ascending=False)
conv_df.to_csv('../tables/GSEA_convergent_pathways.csv', index=False)
print(f"Convergent pathways (>=2 datasets): {len(conv_df)}")
print(f"Significant in all 3: {len(conv_df[conv_df['Datasets']==3])}")